In [1]:
import pandas as pd
import requests
import json
import os
import boto3
import re
from dotenv import load_dotenv

In [2]:
# Villes à analyser
cities = ["Mont Saint Michel",
    "St Malo",
    "Bayeux",
    "Le Havre",
    "Rouen",
    "Paris",
    "Amiens",
    "Lille",
    "Strasbourg",
    "Chateau du Haut Koenigsbourg",
    "Colmar",
    "Eguisheim",
    "Besancon",
    "Dijon",
    "Annecy",
    "Grenoble",
    "Lyon",
    "Rougon",#"Gorges du Verdon", # Probleme avec ce nom de ville
    "Bormes les Mimosas",
    "Cassis",
    "Marseille",
    "Aix en Provence",
    "Avignon",
    "Uzes",
    "Nimes",
    "Aigues Mortes",
    "Saintes Maries de la mer",
    "Collioure",
    "Carcassonne",
    "Ariege",
    "Toulouse",
    "Montauban",
    "Biarritz",
    "Bayonne",
    "La Rochelle"]


In [3]:
# Get the latitude and longitude of the cities
cities_location = []

headers = {
    "User-Agent": "QHA"
}

for city in cities:
    r_location = requests.get(f"https://nominatim.openstreetmap.org/search?format=json&city={city}", headers=headers)
    location_data = r_location.json()

    current_city = {
        "name": location_data[0]['name'],
        "lat": location_data[0]['lat'],
        "lon": location_data[0]['lon'],
    }

    cities_location.append(current_city)
print(cities_location)



[{'name': 'Mont-Saint-Michel', 'lat': '46.7798558', 'lon': '-75.3362610'}, {'name': 'St. Malo', 'lat': '49.3146950', 'lon': '-96.9538228'}, {'name': 'Bayeux', 'lat': '49.2764624', 'lon': '-0.7024738'}, {'name': 'Le Havre', 'lat': '49.4938975', 'lon': '0.1079732'}, {'name': 'Rouen', 'lat': '49.4404591', 'lon': '1.0939658'}, {'name': 'Paris', 'lat': '48.8588897', 'lon': '2.3200410'}, {'name': 'Amiens', 'lat': '49.8941708', 'lon': '2.2956951'}, {'name': 'Lille', 'lat': '50.6365654', 'lon': '3.0635282'}, {'name': 'Strasbourg', 'lat': '48.5846140', 'lon': '7.7507127'}, {'name': 'Château du Haut-Kœnigsbourg', 'lat': '48.2495226', 'lon': '7.3454923'}, {'name': 'Colmar', 'lat': '48.0777517', 'lon': '7.3579641'}, {'name': 'Eguisheim', 'lat': '48.0447968', 'lon': '7.3079618'}, {'name': 'Besançon', 'lat': '47.2380222', 'lon': '6.0243622'}, {'name': 'Dijon', 'lat': '47.3215806', 'lon': '5.0414701'}, {'name': 'Annecy', 'lat': '45.8992348', 'lon': '6.1288847'}, {'name': 'Grenoble', 'lat': '45.187560

In [4]:
df_location = pd.DataFrame(cities_location)
df_location

,name,lat,lon
0,Mont-Saint-Michel,46.7798558,-75.3362610
1,St. Malo,49.3146950,-96.9538228
2,Bayeux,49.2764624,-0.7024738
3,Le Havre,49.4938975,0.1079732
4,Rouen,49.4404591,1.0939658
5,Paris,48.8588897,2.3200410
6,Amiens,49.8941708,2.2956951
7,Lille,50.6365654,3.0635282
8,Strasbourg,48.5846140,7.7507127
9,Château du Haut-Kœnigsbourg,48.2495226,7.3454923


In [5]:
load_dotenv()
api_weather_key = os.getenv("OPENWEATHER_API_KEY")

params = {
    "appid": api_weather_key,
    "units": "metric",
    "lang": "fr",
    "cnt": 5,
}

lat = 44.8333
lon = -0.5667
cnt = 16

url_test = f"https://api.openweathermap.org/data/2.5/forecast?lat=44.8333&lon=-0.5667&units=metric&lang=fr&cnt=7&appid={api_weather_key}"

r = requests.get(url_test)
data = r.json()

data['list']



[{'dt': 1747656000,
  'main': {'temp': 19.19,
   'feels_like': 19.31,
   'temp_min': 19.19,
   'temp_max': 21.31,
   'pressure': 1013,
   'sea_level': 1013,
   'grnd_level': 1008,
   'humidity': 82,
   'temp_kf': -2.12},
  'weather': [{'id': 804,
    'main': 'Clouds',
    'description': 'couvert',
    'icon': '04d'}],
  'clouds': {'all': 100},
  'wind': {'speed': 5.29, 'deg': 250, 'gust': 6.92},
  'visibility': 10000,
  'pop': 0.73,
  'sys': {'pod': 'd'},
  'dt_txt': '2025-05-19 12:00:00'},
 {'dt': 1747666800,
  'main': {'temp': 19.85,
   'feels_like': 19.85,
   'temp_min': 19.85,
   'temp_max': 21.18,
   'pressure': 1013,
   'sea_level': 1013,
   'grnd_level': 1008,
   'humidity': 75,
   'temp_kf': -1.33},
  'weather': [{'id': 804,
    'main': 'Clouds',
    'description': 'couvert',
    'icon': '04d'}],
  'clouds': {'all': 88},
  'wind': {'speed': 5.47, 'deg': 246, 'gust': 6.07},
  'visibility': 10000,
  'pop': 0,
  'sys': {'pod': 'd'},
  'dt_txt': '2025-05-19 15:00:00'},
 {'dt': 1747

In [15]:
all_weather = []
weather_params = {
    "appid": api_weather_key,
    "units": "metric",
    "lang": "fr",
    "cnt": 7,
}

for index,weather in enumerate(cities_location):
    temp_score = 0
    rain = 0
    wind = 0
    wind_score = 0
    rain_score = 0
    
    r_weather = requests.get(f"https://api.openweathermap.org/data/2.5/forecast?lat={weather['lat']}&lon={weather['lon']}", params=weather_params)
    current_weather = r_weather.json()['list']

    for i in range(len(current_weather)):
        temp_score = round(temp_score + (current_weather[i]['main']['feels_like'] / len(current_weather)), 2)
        rain = current_weather[i].get('rain', 0)
        
        if rain != 0:
            rain_key = list(current_weather[i]['rain'].keys())[0]
            match = re.search(r'\d+', rain_key)
            hours = 0

            if match:
                hours = int(match.group())

            rain_daily_score = hours * current_weather[i]['rain'][rain_key]
            rain_score = round(rain_score + (rain_daily_score / len(current_weather)), 2)
        else:
            rain_score = rain_score + 0

        wind = round(wind + (current_weather[i]['wind']['speed'] / len(current_weather)), 2)

        beaufort_score = 0 # Beaufort scale
        if current_weather[i]['wind']['speed'] == 0:
            beaufort_score = 0
        elif current_weather[i]['wind']['speed'] < 5:
            beaufort_score = 1
        elif current_weather[i]['wind']['speed'] < 11:
            beaufort_score = 2
        elif current_weather[i]['wind']['speed'] < 19:
            beaufort_score = 3
        elif current_weather[i]['wind']['speed'] < 28:
            beaufort_score = 4
        elif current_weather[i]['wind']['speed'] < 38:
            beaufort_score = 5
        elif current_weather[i]['wind']['speed'] < 49:
            beaufort_score = 6
        elif current_weather[i]['wind']['speed'] < 61:
            beaufort_score = 7
        elif current_weather[i]['wind']['speed'] < 74:
            beaufort_score = 8
        elif current_weather[i]['wind']['speed'] < 88:
            beaufort_score = 9
        elif current_weather[i]['wind']['speed'] < 102:
            beaufort_score = 10
        elif current_weather[i]['wind']['speed'] < 117:
            beaufort_score = 11
        elif current_weather[i]['wind']['speed'] > 117:
            beaufort_score = 12

        if current_weather[i]['main']['feels_like'] < 25 or current_weather[i]['wind']['speed'] > 30:
            wind_penalty = beaufort_score
        else:
            wind_penalty = 0

        wind_score = round(wind_score + (wind_penalty / len(current_weather)), 2)

    weather_score = round(temp_score - rain_score - wind_score, 3)


    current_weather_data = {
        "index" : index,
        "name" : weather['name'],
        "temperature_mean" : temp_score,
        "rain_mean" : rain_score,
        "wind_score" : wind_score,
        "score" : weather_score,
    }
    all_weather.append(current_weather_data)
       
all_weather


[{'index': 0,
  'name': 'Mont-Saint-Michel',
  'temperature_mean': 3.98,
  'rain_mean': 0,
  'wind_score': 1.58,
  'score': 2.4},
 {'index': 1,
  'name': 'St. Malo',
  'temperature_mean': 5.16,
  'rain_mean': 1.06,
  'wind_score': 1.73,
  'score': 2.37},
 {'index': 2,
  'name': 'Bayeux',
  'temperature_mean': 13.37,
  'rain_mean': 0,
  'wind_score': 1.58,
  'score': 11.79},
 {'index': 3,
  'name': 'Le Havre',
  'temperature_mean': 14.17,
  'rain_mean': 0,
  'wind_score': 1.58,
  'score': 12.59},
 {'index': 4,
  'name': 'Rouen',
  'temperature_mean': 15.07,
  'rain_mean': 0,
  'wind_score': 1.13,
  'score': 13.94},
 {'index': 5,
  'name': 'Paris',
  'temperature_mean': 15.89,
  'rain_mean': 0,
  'wind_score': 0.98,
  'score': 14.91},
 {'index': 6,
  'name': 'Amiens',
  'temperature_mean': 14.05,
  'rain_mean': 0,
  'wind_score': 1.13,
  'score': 12.92},
 {'index': 7,
  'name': 'Lille',
  'temperature_mean': 14.84,
  'rain_mean': 0,
  'wind_score': 0.98,
  'score': 13.86},
 {'index': 8,


In [16]:
df_weather = pd.DataFrame(all_weather)
df_weather = df_weather.sort_values(by='score', ascending=False)
df_weather

,index,name,temperature_mean,rain_mean,wind_score,score
27,27,Collioure,18.91,0.89,1.28,16.74
12,12,Besançon,18.13,0.54,0.98,16.61
8,8,Strasbourg,17.04,0.00,0.98,16.06
16,16,Lyon,17.72,0.45,1.28,15.99
21,21,Aix-en-Provence,18.67,1.81,1.13,15.73
10,10,Colmar,17.75,1.20,0.98,15.57
22,22,Avignon,19.17,2.39,1.28,15.50
18,18,Bormes-les-Mimosas,17.92,1.07,1.43,15.42
15,15,Grenoble,19.70,3.38,0.98,15.34
11,11,Eguisheim,17.54,1.26,0.98,15.30


In [18]:
df_weather.to_csv("export/weather.csv", index=False)